# 핫보드 회의 테이블 조회 (KPI_W / ktws.FCT_HBOARD_MEETING)

이 테이블을 직접 보기 위한 노트북이다. **조회만 한다** — DDL/DML은 넣지 않는다.

## 왜 만들었나

대시보드 로그에 이 경고가 요청마다 찍혔다.

```
[dashboard-cache] hotboard_meeting watermark 확인 실패: Invalid column name 'ETL_TIMESTAMP'.
```

결과 캐시는 시간이 아니라 **데이터가 바뀌었는지**로 갱신 여부를 정한다. 각 소스 테이블의
`MAX(ETL_TIMESTAMP)`를 읽어 묶은 지문을 캐시 키에 넣고, ETL이 새 데이터를 넣으면 지문이
바뀌어 자동으로 다시 조회한다.

이 테이블에는 그 컬럼이 **`ETL_TIMETAMP`** 로 들어가 있다 — `S`가 빠졌다.
그래서 워터마크 조회가 실패하고, `server/dashboardDataFreshness.js`가 이를 잡아
`{status:'unavailable', watermark:null}` 이라는 **상수**를 돌려준다. 지문이 영영
안 바뀌므로 데이터 기반 무효화가 죽고 TTL 1시간만 남는다. 실패는 캐시되지 않아
요청마다 같은 조회를 다시 시도한다.

2026-08-05 실측: `ktws` 스키마 31개 테이블이 `ETL_TIMESTAMP`를 쓴다. 31 대 1이라 오타로 본다.

아래 3번 셀이 그 대조를 다시 보여준다.

## 0. 준비

In [ ]:
# 필요시에만 실행
# %pip install pyodbc pandas python-dotenv

## 1. 접속

자격 증명은 노트북에 적지 않는다 — 대시보드와 같은 `.env`(`Fabric_ID` / `Fabric_PW`)에서 읽는다.
`.env`가 없으면 `ActiveDirectoryInteractive`(브라우저 로그인)로 넘어간다.

In [ ]:
import os
import pyodbc
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv

# 이 노트북 위치에서 대시보드 .env 를 찾는다. 경로가 다르면 ENV_PATH 를 직접 지정할 것.
ENV_PATH = Path.cwd()
for _ in range(5):
    if (ENV_PATH / "main" / "dashboard" / ".env").exists():
        ENV_PATH = ENV_PATH / "main" / "dashboard" / ".env"
        break
    if (ENV_PATH / ".env").exists() and (ENV_PATH / "server" / "fabricClient.js").exists():
        ENV_PATH = ENV_PATH / ".env"
        break
    ENV_PATH = ENV_PATH.parent
load_dotenv(ENV_PATH if ENV_PATH.is_file() else None)

# KPI_W 는 BP_KTWS 엔드포인트에 있다 (server/fabricClient.js 의 DB_TO_SYSTEM 참고)
SERVER = "REPLACE_ME.datawarehouse.fabric.microsoft.com"
DATABASE = "KPI_W"

uid, pwd = os.getenv("Fabric_ID"), os.getenv("Fabric_PW")
if uid and pwd:
    auth = f"Authentication=ActiveDirectoryPassword;UID={uid};PWD={pwd};"
    print(f"자격 증명 사용: {uid}")
else:
    auth = "Authentication=ActiveDirectoryInteractive;"
    print("Fabric_ID/PW 없음 → 브라우저 로그인으로 진행")

conn = pyodbc.connect(
    "DRIVER={ODBC Driver 17 for SQL Server};"
    f"SERVER={SERVER},1433;DATABASE={DATABASE};"
    "Encrypt=yes;TrustServerCertificate=no;Connection Timeout=30;" + auth
)
pd.read_sql("SELECT DB_NAME() AS db, SYSDATETIME() AS now", conn)

## 2. 컬럼 목록

맨 아래 `ETL_TIMETAMP` 를 확인할 것.

In [ ]:
sql = """
SELECT ORDINAL_POSITION AS 순서, COLUMN_NAME AS 컬럼, DATA_TYPE AS 타입,
       CHARACTER_MAXIMUM_LENGTH AS 길이, IS_NULLABLE AS NULL허용
  FROM INFORMATION_SCHEMA.COLUMNS
 WHERE TABLE_SCHEMA = 'ktws' AND TABLE_NAME = 'FCT_HBOARD_MEETING'
 ORDER BY ORDINAL_POSITION
"""
pd.read_sql(sql, conn)

## 3. ETL 컬럼명 — 무엇이 표준인가

31 대 1이면 오타로 봐야 한다. 타입은 셋 다 `datetime2`로 같다.

`EXTRACTION_TIME.ETL_TIME` 은 다른 얘기다 — 추출 시각을 다루는 메타 테이블이라
의도적으로 다를 수 있고, 대시보드 의존성 목록에 들어 있지도 않다.

In [ ]:
sql_dist = """
SELECT COLUMN_NAME AS 컬럼명, DATA_TYPE AS 타입, COUNT(*) AS 테이블수
  FROM INFORMATION_SCHEMA.COLUMNS
 WHERE TABLE_SCHEMA = 'ktws' AND COLUMN_NAME LIKE 'ETL%'
 GROUP BY COLUMN_NAME, DATA_TYPE
 ORDER BY 테이블수 DESC
"""
sql_odd = """
SELECT COLUMN_NAME AS 컬럼명, TABLE_NAME AS 테이블
  FROM INFORMATION_SCHEMA.COLUMNS
 WHERE TABLE_SCHEMA = 'ktws' AND COLUMN_NAME LIKE 'ETL%' AND COLUMN_NAME <> 'ETL_TIMESTAMP'
 ORDER BY COLUMN_NAME, TABLE_NAME
"""
display(pd.read_sql(sql_dist, conn))
display(pd.read_sql(sql_odd, conn))

## 4. 규모와 기간

워터마크로 쓰려던 값(`MAX(ETL_TIMETAMP)`)이 실제로 어떤 값인지도 같이 본다.

In [ ]:
sql = """
SELECT COUNT(*)                AS 행수,
       COUNT(DISTINCT lead_id) AS 리드수,
       MIN(meet_dt)            AS 최초_회의일,
       MAX(meet_dt)            AS 최종_회의일,
       MIN(ETL_TIMETAMP)       AS 최초_적재,
       MAX(ETL_TIMETAMP)       AS 최종_적재
  FROM ktws.FCT_HBOARD_MEETING
"""
pd.read_sql(sql, conn)

## 5. 샘플 행

컬럼이 36개라 한눈에 안 들어온다. 자주 보는 것만 먼저 뽑는다.
전체를 보려면 아래 `SELECT TOP 20 *` 쪽을 쓸 것.

In [ ]:
sql = """
SELECT TOP 50
       meet_dt, meet_ym_seq, BRAND, meet_status_nm, lead_id,
       contact_nm, owner_nm, chip_status_nm, contract_ratio,
       comp_brand_nm, comp_model_nm, own_brand_nm, own_model_nm,
       close_yn, close_dt, ETL_TIMETAMP
  FROM ktws.FCT_HBOARD_MEETING
 ORDER BY meet_dt DESC, meet_ym_seq DESC
"""
pd.read_sql(sql, conn)

In [ ]:
pd.set_option("display.max_columns", None)
pd.read_sql("SELECT TOP 20 * FROM ktws.FCT_HBOARD_MEETING ORDER BY meet_dt DESC", conn)

## 6. 회차(`meet_ym_seq`) 분포

인증 리포트 `hotboard_meeting` 의 `meet_round` 파라미터가 이 컬럼에 걸린다.
**정수**다 — 챗봇이 `"3회차"` 를 그대로 넘겨 오류 없이 0행이 나온 적이 있다
(2026-08-05, 평가 No.47). 지금은 파이프라인에서 숫자만 남기도록 고쳤다.

In [ ]:
sql = """
SELECT YEAR(meet_dt) AS 연, MONTH(meet_dt) AS 월, meet_ym_seq AS 회차, COUNT(*) AS 건수
  FROM ktws.FCT_HBOARD_MEETING
 WHERE meet_dt IS NOT NULL
 GROUP BY YEAR(meet_dt), MONTH(meet_dt), meet_ym_seq
 ORDER BY 연 DESC, 월 DESC, 회차
"""
pd.read_sql(sql, conn)

## 7. 챗봇이 실제로 돌린 조건 재현

평가 No.47 — "2026년 4월 3회차 미팅 진행한 이력은 총 몇건인지".
리포트는 여기에 조직 필터와 SC 조인을 더 걸지만, 건수만 보려면 이걸로 충분하다.

In [ ]:
sql = """
SELECT COUNT(*) AS 건수
  FROM ktws.FCT_HBOARD_MEETING
 WHERE YEAR(meet_dt) = 2026 AND MONTH(meet_dt) = 4 AND meet_ym_seq = 3
"""
print("meet_ym_seq = 3      :", pd.read_sql(sql, conn).iloc[0, 0])

# 값을 문자열 그대로 넘기면 어떻게 되는지 — 오류 없이 0이 나온다
sql_bad = """
SELECT COUNT(*) AS 건수
  FROM ktws.FCT_HBOARD_MEETING
 WHERE YEAR(meet_dt) = 2026 AND MONTH(meet_dt) = 4
   AND CAST(meet_ym_seq AS NVARCHAR(10)) = N'3회차'
"""
print("meet_ym_seq = '3회차' :", pd.read_sql(sql_bad, conn).iloc[0, 0])

## 8. 자유 조회

아래 `sql` 만 바꿔 쓰면 된다. 조회 외의 문(INSERT/UPDATE/DELETE/DDL)은 넣지 말 것.

In [ ]:
sql = """
SELECT TOP 100 *
  FROM ktws.FCT_HBOARD_MEETING
 ORDER BY ETL_TIMETAMP DESC
"""
pd.read_sql(sql, conn)